# Day 12 — Gold Dimensional: Dimension Tables

Builds all 13 dimension tables from Silver sources.
Surrogate keys are generated with `monotonically_increasing_id()`.
SCD Type 2 dimensions (DimCharger, DimCustomer, DimTariff) use `is_current` + `valid_from`/`valid_to` derived from Silver `updated_at` and `valid_from`/`valid_to`.
Fields not present in source are omitted (no nulls fabricated).

| Dimension | Source Silver entity | SCD |
|---|---|---|
| DimCountry | states | Type 1 |
| DimState | states | Type 1 |
| DimCity | cities | Type 1 |
| DimStation | stations | Type 1 |
| DimCharger | chargers + realtime charging_sessions | Type 2 |
| DimCustomer | customers | Type 2 |
| DimVehicle | vehicles | Type 1 |
| DimEmployee | employees | Type 1 |
| DimFranchisePartner | partners | Type 1 |
| DimChargeCard | charge_cards | Type 1 |
| DimTariff | tariffs | Type 2 |
| DimWeather | weather + cities | Type 1 |
| DimTime | generated (2020–2030) | — |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

print('Imports OK')

In [ ]:
# Cell 2 — Constants
SILVER_API = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
SILVER_RT  = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime'
GOLD_DIM   = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/dims'
PIPELINE   = 'pl_gold_dimensional_dims_v1'
RUN_TS     = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def read_silver(name):
    return spark.read.format('delta').load(f'{SILVER_API}/{name}')

def write_dim(df, dim_name):
    path = f'{GOLD_DIM}/{dim_name}'
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(path)
    print(f'  {dim_name:<25} {df.count():>8} rows  -> {path}')

print(f'Gold dims : {GOLD_DIM}')
print(f'RUN_TS    : {RUN_TS}')

In [ ]:
# Cell 3 — DimCountry
# Source: states (country column); deduplicated to unique countries.
# country_code omitted — not in source.
states_raw = read_silver('states')

dim_country = (
    states_raw
    .select(F.trim(F.col('country')).alias('country_name'))
    .distinct()
    .filter(F.col('country_name').isNotNull() & (F.col('country_name') != ''))
    .withColumn('country_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('country_key', 'country_name', 'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_country, 'DimCountry')

In [ ]:
# Cell 4 — DimState
# Source: states
dim_state = (
    states_raw
    .select(
        F.trim(F.col('state_code')).alias('state_code'),
        F.trim(F.col('name')).alias('state_name'),
        F.trim(F.col('country')).alias('country_name'),
    )
    .filter(F.col('state_code').isNotNull() & (F.col('state_code') != ''))
    .distinct()
    .withColumn('state_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('state_key', 'state_code', 'state_name', 'country_name',
            'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_state, 'DimState')

In [ ]:
# Cell 5 — DimCity
# Source: cities. postcode omitted — not in source.
cities_raw = read_silver('cities')

dim_city = (
    cities_raw
    .select(
        F.col('city_id'),
        F.trim(F.col('name')).alias('city_name'),
        F.trim(F.col('state_code')).alias('state_code'),
        F.trim(F.col('country')).alias('country_name'),
        F.col('latitude').alias('city_lat'),
        F.col('longitude').alias('city_lon'),
        F.col('population'),
    )
    .filter(F.col('city_id').isNotNull())
    .withColumn('city_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('city_key', 'city_id', 'city_name', 'state_code', 'country_name',
            'city_lat', 'city_lon', 'population', 'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_city, 'DimCity')

In [ ]:
# Cell 6 — DimStation
# Source: stations. site_type omitted — not in source.
stations_raw = read_silver('stations')

dim_station = (
    stations_raw
    .select(
        F.col('station_id'),
        F.trim(F.col('name')).alias('station_name'),
        F.trim(F.col('city')).alias('city_name'),
        F.trim(F.col('state')).alias('state_code'),
        F.trim(F.col('country')).alias('country_name'),
        F.col('latitude').alias('lat'),
        F.col('longitude').alias('lng'),
        F.col('total_chargers'),
        F.trim(F.col('status')).alias('status'),
    )
    .filter(F.col('station_id').isNotNull())
    .withColumn('station_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('station_key', 'station_id', 'station_name', 'city_name',
            'state_code', 'country_name', 'lat', 'lng',
            'total_chargers', 'status', 'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_station, 'DimStation')

In [ ]:
# Cell 7 — DimCharger (SCD Type 2)
# Source: chargers (base) + realtime charging_sessions (firmware_ver per charger, latest).
# SCD2: each row is a version; is_current=True for the latest.
# valid_from = charger created_at, valid_to = NULL for current row.
chargers_raw = read_silver('chargers')
rt_sessions  = spark.read.format('delta').load(f'{SILVER_RT}/charging_sessions')

# Latest firmware version seen per charger from realtime telemetry
fw_win = Window.partitionBy('charger_id').orderBy(F.col('plug_in_ts').desc())
latest_fw = (
    rt_sessions
    .filter(F.col('firmware_ver').isNotNull())
    .withColumn('_rn', F.row_number().over(fw_win)).filter(F.col('_rn') == 1).drop('_rn')
    .select(F.col('charger_id'), F.col('firmware_ver').alias('firmware_version'))
)

dim_charger = (
    chargers_raw
    .join(latest_fw, on='charger_id', how='left')
    .select(
        F.col('charger_id'),
        F.col('station_id'),
        F.trim(F.col('charger_type')).alias('charger_type'),
        F.trim(F.col('connector_type')).alias('connector_type'),
        F.col('power_kw').alias('max_kw'),
        F.trim(F.col('status')).alias('status'),
        F.col('firmware_version'),
        F.col('created_at').alias('valid_from'),
        F.lit(None).cast('timestamp').alias('valid_to'),
        F.lit(True).alias('is_current'),
    )
    .filter(F.col('charger_id').isNotNull())
    .withColumn('charger_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('charger_key', 'charger_id', 'station_id', 'charger_type',
            'connector_type', 'max_kw', 'status', 'firmware_version',
            'valid_from', 'valid_to', 'is_current',
            'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_charger, 'DimCharger')

In [ ]:
# Cell 8 — DimCustomer (SCD Type 2)
# Source: customers. loyalty_tier omitted — not in source.
# signup_date derived from created_at.
customers_raw = read_silver('customers')

dim_customer = (
    customers_raw
    .select(
        F.col('customer_id'),
        F.trim(F.col('first_name')).alias('first_name'),
        F.trim(F.col('last_name')).alias('last_name'),
        F.concat_ws(' ', F.trim('first_name'), F.trim('last_name')).alias('full_name'),
        F.trim(F.col('email')).alias('email'),
        F.trim(F.col('phone')).alias('phone'),
        F.trim(F.col('city')).alias('city'),
        F.trim(F.col('state')).alias('state'),
        F.trim(F.col('country')).alias('country'),
        F.to_date(F.col('created_at')).alias('signup_date'),
        F.col('created_at').alias('valid_from'),
        F.lit(None).cast('timestamp').alias('valid_to'),
        F.lit(True).alias('is_current'),
    )
    .filter(F.col('customer_id').isNotNull())
    .withColumn('customer_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('customer_key', 'customer_id', 'first_name', 'last_name', 'full_name',
            'email', 'phone', 'city', 'state', 'country', 'signup_date',
            'valid_from', 'valid_to', 'is_current',
            'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_customer, 'DimCustomer')

In [ ]:
# Cell 9 — DimVehicle
# Source: vehicles. vin omitted — not in source.
vehicles_raw = read_silver('vehicles')

dim_vehicle = (
    vehicles_raw
    .select(
        F.col('vehicle_id'),
        F.col('customer_id'),
        F.trim(F.col('make')).alias('make'),
        F.trim(F.col('model')).alias('model'),
        F.col('year'),
        F.col('battery_capacity').alias('battery_capacity_kwh'),
        F.col('range_km'),
        F.trim(F.col('license_plate')).alias('license_plate'),
    )
    .filter(F.col('vehicle_id').isNotNull())
    .withColumn('vehicle_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('vehicle_key', 'vehicle_id', 'customer_id', 'make', 'model',
            'year', 'battery_capacity_kwh', 'range_km', 'license_plate',
            'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_vehicle, 'DimVehicle')

In [ ]:
# Cell 10 — DimEmployee
# Source: employees
employees_raw = read_silver('employees')

dim_employee = (
    employees_raw
    .select(
        F.col('employee_id'),
        F.trim(F.col('first_name')).alias('first_name'),
        F.trim(F.col('last_name')).alias('last_name'),
        F.concat_ws(' ', F.trim('first_name'), F.trim('last_name')).alias('full_name'),
        F.trim(F.col('email')).alias('email'),
        F.trim(F.col('role')).alias('role'),
        F.trim(F.col('department')).alias('department'),
        F.col('station_id'),
        F.col('hire_date'),
    )
    .filter(F.col('employee_id').isNotNull())
    .withColumn('employee_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('employee_key', 'employee_id', 'first_name', 'last_name', 'full_name',
            'email', 'role', 'department', 'station_id', 'hire_date',
            'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_employee, 'DimEmployee')

In [ ]:
# Cell 11 — DimFranchisePartner
# Source: partners
partners_raw = read_silver('partners')

dim_partner = (
    partners_raw
    .select(
        F.col('partner_id'),
        F.trim(F.col('name')).alias('partner_name'),
        F.trim(F.col('country')).alias('country'),
        F.trim(F.col('contact_email')).alias('contact_email'),
        F.trim(F.col('status')).alias('status'),
    )
    .filter(F.col('partner_id').isNotNull())
    .withColumn('partner_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('partner_key', 'partner_id', 'partner_name', 'country',
            'contact_email', 'status', 'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_partner, 'DimFranchisePartner')

In [ ]:
# Cell 12 — DimChargeCard
# Source: charge_cards. rfid_uid omitted — not in source.
# card_number_masked: keep first 4 + last 4, mask the middle.
cards_raw = read_silver('charge_cards')

dim_card = (
    cards_raw
    .select(
        F.col('card_id'),
        F.col('customer_id'),
        # Mask card_number: show first 4 and last 4 only
        F.when(
            F.col('card_number').isNotNull() & (F.length('card_number') >= 8),
            F.concat(
                F.substring('card_number', 1, 4),
                F.lit('-****-****-'),
                F.substring('card_number', -4, 4)
            )
        ).otherwise(F.lit('****')).alias('card_number_masked'),
        F.trim(F.col('status')).alias('status'),
        F.col('issued_at'),
        F.col('expires_at'),
    )
    .filter(F.col('card_id').isNotNull())
    .withColumn('card_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('card_key', 'card_id', 'customer_id', 'card_number_masked',
            'status', 'issued_at', 'expires_at',
            'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_card, 'DimChargeCard')

In [ ]:
# Cell 13 — DimTariff (SCD Type 2)
# Source: tariffs. peak_offpeak omitted — not in source.
# is_current = valid_to IS NULL OR valid_to > now()
tariffs_raw = read_silver('tariffs')

dim_tariff = (
    tariffs_raw
    .select(
        F.col('tariff_id'),
        F.trim(F.col('name')).alias('tariff_name'),
        F.col('price_per_kwh').alias('rate_per_kwh'),
        F.col('price_per_min').alias('rate_per_min'),
        F.trim(F.col('currency')).alias('currency'),
        F.col('valid_from').alias('effective_from'),
        F.col('valid_to').alias('effective_to'),
        F.when(
            F.col('valid_to').isNull() | (F.col('valid_to') > F.lit(RUN_TS).cast('timestamp')),
            F.lit(True)
        ).otherwise(F.lit(False)).alias('is_current'),
    )
    .filter(F.col('tariff_id').isNotNull())
    .withColumn('tariff_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('tariff_key', 'tariff_id', 'tariff_name', 'rate_per_kwh', 'rate_per_min',
            'currency', 'effective_from', 'effective_to', 'is_current',
            'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_tariff, 'DimTariff')

In [ ]:
# Cell 14 — DimWeather
# Source: weather joined to cities for city_id → city_name + state_code
weather_raw = read_silver('weather')
cities_raw2 = read_silver('cities')

city_lookup = cities_raw2.select(
    F.col('city_id'),
    F.trim(F.col('name')).alias('city_name'),
    F.trim(F.col('state_code')).alias('state_code'),
)

dim_weather = (
    weather_raw
    .join(city_lookup, on='city_id', how='left')
    .select(
        F.col('city_id'),
        F.col('city_name'),
        F.col('state_code'),
        F.col('temperature_c'),
        F.col('humidity_pct'),
        F.col('wind_speed_kmh'),
        F.trim(F.col('condition')).alias('condition'),
        F.col('recorded_at'),
    )
    .filter(F.col('city_id').isNotNull())
    .withColumn('weather_key', F.monotonically_increasing_id())
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('weather_key', 'city_id', 'city_name', 'state_code',
            'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'condition', 'recorded_at',
            'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_weather, 'DimWeather')

In [ ]:
# Cell 15 — DimTime (generated, hour grain, 2020–2030)
# No source data needed — fully computed.
# financial_year_au: AU financial year starts 1 July (e.g. Jul 2025–Jun 2026 = FY2026)

# Generate one row per hour from 2020-01-01 00:00 to 2030-12-31 23:00
hours_per_year = 365 * 24
total_hours    = hours_per_year * 11  # 2020–2030 inclusive

dim_time = (
    spark.range(total_hours)
    .withColumn('ts', (F.lit('2020-01-01 00:00:00').cast('timestamp')
                       + (F.col('id') * 3600).cast('long').cast('interval')).cast('timestamp'))
    # Simpler: use date_add + hour arithmetic
)

# Rebuild using SQL date arithmetic (serverless-safe)
dim_time = (
    spark.range(total_hours)
    .select(
        F.col('id').alias('hour_offset'),
        F.expr("timestamp '2020-01-01 00:00:00' + make_interval(0, 0, 0, 0, cast(id as int), 0, 0)").alias('ts')
    )
    .select(
        F.col('ts'),
        F.to_date('ts').alias('date'),
        F.year('ts').alias('year'),
        F.month('ts').alias('month'),
        F.dayofmonth('ts').alias('day'),
        F.hour('ts').alias('hour'),
        F.date_format('ts', 'EEEE').alias('day_of_week'),
        F.dayofweek('ts').alias('day_of_week_num'),   # 1=Sun, 7=Sat
        F.when(F.dayofweek('ts').isin(1, 7), F.lit(True)).otherwise(F.lit(False)).alias('is_weekend'),
        F.quarter('ts').alias('quarter'),
        # AU financial year: Jul–Jun, label by the ending calendar year
        F.when(F.month('ts') >= 7,
            F.concat(F.lit('FY'), (F.year('ts') + 1).cast('string'))
        ).otherwise(
            F.concat(F.lit('FY'), F.year('ts').cast('string'))
        ).alias('financial_year_au'),
    )
    .withColumn('time_key', F.concat_ws('',
        F.col('year').cast('string'),
        F.lpad(F.col('month').cast('string'), 2, '0'),
        F.lpad(F.col('day').cast('string'),   2, '0'),
        F.lpad(F.col('hour').cast('string'),  2, '0')
    ))  # e.g. 2026071409
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select('time_key', 'ts', 'date', 'year', 'month', 'day', 'hour',
            'day_of_week', 'day_of_week_num', 'is_weekend', 'quarter',
            'financial_year_au', 'gold_ingested_at', 'gold_pipeline')
)

write_dim(dim_time, 'DimTime')

In [ ]:
# Cell 16 — Summary
dims = [
    'DimCountry', 'DimState', 'DimCity', 'DimStation', 'DimCharger',
    'DimCustomer', 'DimVehicle', 'DimEmployee', 'DimFranchisePartner',
    'DimChargeCard', 'DimTariff', 'DimWeather', 'DimTime'
]
print('=' * 55)
print('GOLD DIMENSIONAL — DIM TABLES SUMMARY')
print('=' * 55)
for d in dims:
    path = f'{GOLD_DIM}/{d}'
    try:
        cnt = spark.read.format('delta').load(path).count()
        print(f'  {d:<25} {cnt:>8} rows')
    except Exception as e:
        print(f'  {d:<25} ERROR: {e}')
print(f'\nRUN_TS : {RUN_TS}')